In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    sum,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_creditm_data(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
            # "CreditMDataId String"
            # ",CreditMDataKey String"
            # ",AccountNumber String"
            # ",SourceSystem String"
            # ",IngestionDate timestamp"
            # ",BatchId string"
            # ",PortfolioName String"
            # ",CreditMPortfolio String"
            # ",TTLBalance decimal(13,2)"
            # ",PaymentDate Date"
            # ",LastPaymentDate Date"
            # ",CepalCreditLossAmount decimal(13,2)"
            # ",CepalCreditLossDate Date"
            # ",Payments decimal(13,0)"
            # ",BoGAssetSegment String"
            # ",BoGBucketPreviousQuarter String"
            # ",BoGLegalStatus String"
            # ",BoGBucketCurrentQuarter String"
            # ",ModificationFlagTillPreviousQuarter String"
            # ",ModificationFlagTillCurrentQuarter String"
            # ",SecuredFlag String"
            # ",Bsd34Flag String"
            # ",BondFlag String"
            # ",InterestRate Int"
            # ",Currency String"
            # ",Investor String"
            # ",InvestorCountry String"
            # ",InvestorCountryCode String"
        #  )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.CreditMData"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "creditm",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== Revolving_Debtor_Transaction_History ==================

        Revolving_Debtor_Transaction_History_extra_filters = None

        if Revolving_Debtor_Transaction_History_extra_filters is not None:
            Revolving_Debtor_Transaction_History_extra_filters.append(date_filter)
            Revolving_Debtor_Transaction_History_final_filters = " AND ".join(Revolving_Debtor_Transaction_History_extra_filters)
        else:
            Revolving_Debtor_Transaction_History_final_filters = date_filter

        Revolving_Debtor_Transaction_History_selected_fields = [
            "AccountNumber",
            "TransactionAmount",
            "PaymentDate",
        ]

        Revolving_Debtor_Transaction_History = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "creditm_CepalRevolvingDebtorTransactionHistory",
            Revolving_Debtor_Transaction_History_final_filters,
            Revolving_Debtor_Transaction_History_selected_fields,
        )

        # ================== Legacy_Debtor_Transaction_History ==================

        Legacy_Debtor_Transaction_History_extra_filters = None

        if Legacy_Debtor_Transaction_History_extra_filters is not None:
            Legacy_Debtor_Transaction_History_extra_filters.append(date_filter)
            Legacy_Debtor_Transaction_History_final_filters = " AND ".join(Legacy_Debtor_Transaction_History_extra_filters)
        else:
            Legacy_Debtor_Transaction_History_final_filters = date_filter

        Legacy_Debtor_Transaction_History_selected_fields = [
            "AccountNumber",
            "TransactionAmount",
            "PaymentDate",
        ]

        Legacy_Debtor_Transaction_History = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "creditm_CepalLegacyDebtorTransactionHistory",
            Legacy_Debtor_Transaction_History_final_filters,
            Legacy_Debtor_Transaction_History_selected_fields,
        )

        # ================== Omega_Debtor_Transaction_History ==================

        Omega_Debtor_Transaction_History_extra_filters = None

        if Omega_Debtor_Transaction_History_extra_filters is not None:
            Omega_Debtor_Transaction_History_extra_filters.append(date_filter)
            Omega_Debtor_Transaction_History_final_filters = " AND ".join(Omega_Debtor_Transaction_History_extra_filters)
        else:
            Omega_Debtor_Transaction_History_final_filters = date_filter

        Omega_Debtor_Transaction_History_selected_fields = [
            "AccountNumber",
            "TransactionAmount",
            "PaymentDate",
        ]

        Omega_Debtor_Transaction_History = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "creditm_CepalOmegaDebtorTransactionHistory",
            Omega_Debtor_Transaction_History_final_filters,
            Omega_Debtor_Transaction_History_selected_fields,
        )

        # ================== Revolving_Debtor_Data ==================

        Revolving_Debtor_Data_extra_filters = None

        if Revolving_Debtor_Data_extra_filters is not None:
            Revolving_Debtor_Data_extra_filters.append(date_filter)
            Revolving_Debtor_Data_final_filters = " AND ".join(Revolving_Debtor_Data_extra_filters)
        else:
            Revolving_Debtor_Data_final_filters = date_filter

        Revolving_Debtor_Data_selected_fields = [
            "AccountNumber",
            "TTLBALANCE",
            "LastPaymentDate",
            "CEPAL_CreditLossAmount",
            "CEPAL_CreditLossDate",
            "BusinessDate",
        ]

        Revolving_Debtor_Data = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "creditm_CepalRevolvingDebtorData",
            Revolving_Debtor_Data_final_filters,
            Revolving_Debtor_Data_selected_fields,
        )

        # ================== Legacy_Debtor_Data ==================

        Legacy_Debtor_Data_extra_filters = None

        if Legacy_Debtor_Data_extra_filters is not None:
            Legacy_Debtor_Data_extra_filters.append(date_filter)
            Legacy_Debtor_Data_final_filters = " AND ".join(Legacy_Debtor_Data_extra_filters)
        else:
            Legacy_Debtor_Data_final_filters = date_filter

        Legacy_Debtor_Data_selected_fields = [
            "AccountNumber",
            "CurrentBalance",
            "LastPaymentDate",
            "CEPAL_CreditLossAmount",
            "CEPAL_CreditLossDate",
            "BusinessDate",
        ]

        Legacy_Debtor_Data = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "creditm_CepalLegacyDebtorData",
            Legacy_Debtor_Data_final_filters,
            Legacy_Debtor_Data_selected_fields,
        )

        # ================== Omega_Debtor_Data ==================

        Omega_Debtor_Data_extra_filters = None

        if Omega_Debtor_Data_extra_filters is not None:
            Omega_Debtor_Data_extra_filters.append(date_filter)
            Omega_Debtor_Data_final_filters = " AND ".join(Omega_Debtor_Data_extra_filters)
        else:
            Omega_Debtor_Data_final_filters = date_filter

        Omega_Debtor_Data_selected_fields = [
            "AccountNumber",
            "TTLOUT",
            "LastPaymentDate",
            "CEPAL_CreditLossAmount",
            "CEPAL_CreditLossDate",
            "BusinessDate",
        ]

        Omega_Debtor_Data = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "creditm_CepalOmegaDebtorData",
            Omega_Debtor_Data_final_filters,
            Omega_Debtor_Data_selected_fields,
        )

        # =============== Unions & Joins =============

        # Unions
        Unioned_Debtor_Transaction_History = Revolving_Debtor_Transaction_History.union(Legacy_Debtor_Transaction_History).union(Omega_Debtor_Transaction_History)
        Unioned_Debtor_Transaction_History = Unioned_Debtor_Transaction_History.groupBy("AccountNumber", "PaymentDate").agg(sum("TransactionAmount").alias("TotalTransactionAmount"))

        Unioned_Debtor_Transaction_History = Unioned_Debtor_Transaction_History.withColumn("PaymentsCalc", -coalesce(Unioned_Debtor_Transaction_History.TotalTransactionAmount, lit(0)))

        Revolving_Debtor_Data = Revolving_Debtor_Data.withColumn("CreditMPortfolio", lit("Revolving"))
        Legacy_Debtor_Data = Legacy_Debtor_Data.withColumn("CreditMPortfolio", lit("Legacy"))
        Omega_Debtor_Data = Omega_Debtor_Data.withColumn("CreditMPortfolio", lit("Omega"))

        Unioned_Debtor_Data = Revolving_Debtor_Data.union(Legacy_Debtor_Data).union(Omega_Debtor_Data)

        # Join
        join_df = Unioned_Debtor_Data.join(
            Unioned_Debtor_Transaction_History,
            Unioned_Debtor_Data.AccountNumber == Unioned_Debtor_Transaction_History.AccountNumber,
            "left",
        )

        # =============== Primary Keys ===============
        join_primary_key_fields = [
            Unioned_Debtor_Data.AccountNumber,
            join_df.CreditMPortfolio,
            join_df.PaymentDate,
            lit("creditm"),
        ]

        join_key_fields = [
            Unioned_Debtor_Data.AccountNumber,
            join_df.CreditMPortfolio,
            join_df.PaymentDate,
        ]

        join_df = join_df.withColumn(
            "CreditMDataId",
            sha2(concat_ws(";", *join_primary_key_fields), 256),
        ).withColumn("CreditMDataKey", concat_ws(";", *join_key_fields))

        # ================== Selection of Final Columns ==================

        result_df = join_df.select(
            join_df.CreditMDataId.alias("CreditMDataId"),
            join_df.CreditMDataKey.alias("CreditMDataKey"),
            Unioned_Debtor_Data.AccountNumber.alias("AccountNumber"),
            lit("creditm").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            lit("CreditM").cast("String").alias("PortfolioName"),
            Unioned_Debtor_Data.CreditMPortfolio.cast("string").alias("CreditMPortfolio"),
            Unioned_Debtor_Data.TTLBALANCE.cast("decimal(13,2)").alias("TTLBalance"),
            Unioned_Debtor_Transaction_History.PaymentDate.cast("date").alias("PaymentDate"),
            Unioned_Debtor_Data.LastPaymentDate.cast("date").alias("LastPaymentDate"),
            Unioned_Debtor_Data.CEPAL_CreditLossAmount.cast("decimal(13,2)").alias("CepalCreditLossAmount"),
            Unioned_Debtor_Data.CEPAL_CreditLossDate.cast("date").alias("CepalCreditLossDate"),            
            Unioned_Debtor_Transaction_History.PaymentsCalc.cast("decimal(13,0)").alias("Payments"),
            lit("CL").cast("String").alias("BoGAssetSegment"),
            lit("Denounced").cast("String").alias("BoGBucketPreviousQuarter"),
            lit("Of which in procedure to Obtain Enforceable Title").cast("String").alias("BoGLegalStatus"),
            lit("Denounced").cast("String").alias("BoGBucketCurrentQuarter"),
            lit("No").cast("String").alias("ModificationFlagTillPreviousQuarter"),
            lit("No").cast("String").alias("ModificationFlagTillCurrentQuarter"),
            lit("No").cast("String").alias("SecuredFlag"),
            lit("No").cast("String").alias("Bsd34Flag"),
            lit("No").cast("String").alias("BondFlag"),
            lit(0).cast("Int").alias("InterestRate"),
            lit("EUR").cast("String").alias("Currency"),
            lit("SAMCO HOLDINGS S.A.R.L.").cast("String").alias("Investor"),
            lit("Luxembourg").cast("String").alias("InvestorCountry"),
            lit("LU").cast("String").alias("InvestorCountryCode"),
            Unioned_Debtor_Data.BusinessDate,
        )

        # ================== Append ==================
        result_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_creditm_data(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)